# Explore a sham-feeding `*_lickprocessed.pkl`

Look at everything in a sham-feeding pickle (before NWB conversion). Set `PKL` to file path.

The pickle is a **2-tuple of dicts, one per recording side / hemisphere**. Each side's identity is read from its `Full_side_name` (e.g. `COM3_Left_mNacSh`). Photometry channels: `analog_1` = 470 nm gACh4h, `analog_2` = 565 nm rDA3m, `analog_3` = 405 nm gACh4h reference.

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PKL = "IM1923_Trial-SF5-Sucrose_11-06-2025_lickprocessed.pkl"
PKL = "IM1928_Trial-RF2-Sucrose_10-17-2025_lickprocessed.pkl"

# Per analog channel: (pickle key, excitation wavelength nm, sensor/role, plot color)
CHANNELS = [
    ("analog_1", 470, "gACh4h",     "#2CA02C"),
    ("analog_2", 565, "rDA3m",      "#D62728"),
    ("analog_3", 405, "gACh4h ref", "#8FBF8F"),
]
# The lick-burst stats live under this key (ILI-based bursts, 2000 ms gap threshold)
BURST_KEY = "LickBurst_Vars_BurstDefinitionILI_basedThresh2000"

with open(PKL, "rb") as pkl_file:
    data = pickle.load(pkl_file)


def describe(value):
    """One-line type/shape summary of a value."""
    if isinstance(value, np.ndarray):
        return f"ndarray {value.shape} {value.dtype}"
    if isinstance(value, pd.DataFrame):
        return f"DataFrame {value.shape}"
    if isinstance(value, pd.Series):
        return f"Series len={len(value)} ({value.dtype})"
    if isinstance(value, dict):
        return f"dict keys={list(value.keys())}"
    if isinstance(value, (list, tuple)):
        return f"{type(value).__name__} len={len(value)}"
    return f"{type(value).__name__}: {repr(value)[:70]}"


print("top-level:", type(data).__name__, "| n sides:", len(data))
for index, side_data in enumerate(data):
    print(f"  side {index}: {side_data['Full_side_name']} | {len(side_data)} keys | {len(side_data['analog_1'])} samples")

## Print full key inventory (both sides)

Every key with its type and shape/length, one column per side (recording hemisphere).

In [ ]:
pd.set_option("display.max_rows", None, "display.max_colwidth", 90)
# One column per side; each row is a key with its type/shape summary
pd.DataFrame({side_data["Full_side_name"]: {key: describe(value) for key, value in side_data.items()}
              for side_data in data})

## Session / animal metadata

In [ ]:
scalar_keys = [
    "filename", "subject_ID", "Full_animalNumber", "Sex", "DOB", "Strain",
    "date_time", "end_time", "Full_side_name", "Left_COM", "Right_COM",
    "Hit_L", "Target_L", "Hit_R", "Target_R", "GramConsumed", "GramInPan",
    "mode", "sampling_rate", "n_analog_signals", "n_digital_signals",
    "LED_current", "volts_per_division", "version",
    "BottleIn_frameNum", "SessionStart_frameNum", "SessionEnd_frameNum",
]
for side_data in data:
    print("=" * 70, "\n", side_data["Full_side_name"])
    for key in scalar_keys:
        print(f"  {key:24s} {side_data.get(key)}")
    print("  Config_Lickdetection:", side_data["Processing_params"]["Config_Lickdetection"])

## Photometry signals

`analog_1/2/3` are the raw channels (volts), `*_filt` are pyPhotometry-filtered, `analogN_hampel` are outlier-cleaned. `time` is in ms.

Plots: one figure per processing level (raw / filtered / hampel) per side, each with three subplots (470 signal, 405 reference, 565 rDA3m) — six figures total.

In [ ]:
# Per-channel signal stats
for side_data in data:
    fs = side_data["sampling_rate"]
    time_min = side_data["time"] / 1000 / 60          # ms -> minutes (photometry stream clock)
    print(f"{side_data['Full_side_name']}: {fs} Hz | {len(side_data['analog_1'])} samples | {time_min[-1]:.1f} min")
    for analog_key, wavelength, sensor, _color in CHANNELS:
        channel = side_data[analog_key]
        print(f"   {analog_key} ({wavelength} nm, {sensor:11s}): "
              f"mean={channel.mean():.4f} V  min={channel.min():.4f}  max={channel.max():.4f}")

# Photometry: one figure per (processing level x side), 3 subplots (one per wavelength).
# Levels: raw (analog_N), pyPhotometry-filtered (analog_N_filt), hampel-cleaned (analogN_hampel).
photometry_levels = [
    ("raw",      lambda key: key),
    ("filtered", lambda key: f"{key}_filt"),
    ("hampel",   lambda key: f"analog{key[-1]}_hampel"),
]
ordered_channels = sorted(CHANNELS, key=lambda ch: (470, 405, 565).index(ch[1]))  # 470 signal, its 405 ref, 565 rDA3m
for level_name, key_for in photometry_levels:
    for side_data in data:
        time_min = side_data["time"] / 1000 / 60
        fig, ax = plt.subplots(len(ordered_channels), 1, figsize=(13, 8), sharex=True)
        for axis, (analog_key, wavelength, sensor, color) in zip(ax, ordered_channels):
            axis.plot(time_min, side_data[key_for(analog_key)], color=color, lw=0.6)
            axis.set_ylabel("V"); axis.set_title(f"{wavelength} nm ({sensor})", fontsize=9)
        ax[-1].set_xlabel("time (min)")
        fig.suptitle(f"{level_name} photometry — {side_data['Full_side_name']}")
        fig.tight_layout()

## Sync signals (rsync)

`digital_1` is the rsync TTL; rising edges are at `pulse_inds_1` (samples) / `pulse_times_1` (ms). `Rsync_aligned-from-licks` and `rSync_interpolated_from_video` are the rsync reconstructed from the lick and video streams, used to align those streams to photometry.

In [ ]:
for side_data in data:
    fs = side_data["sampling_rate"]
    print(side_data["Full_side_name"],
          "| digital_1 high:", int(side_data["digital_1"].sum()),
          "| n pulses:", len(side_data["pulse_inds_1"]),
          "| Rsync_aligned-from-licks:", side_data["Rsync_aligned-from-licks"].shape,
          "| rSync_interpolated_from_video:", side_data["rSync_interpolated_from_video"].shape)

    inter_pulse_ms = np.diff(side_data["pulse_times_1"])   # rsync inter-pulse intervals (ms)
    # bin on the 86 Hz sample grid: one bin per integer sample gap (1000/fs ms), so the
    # +/-1-sample digitization jitter lands on discrete, comparable bins
    sample_ms = 1000.0 / fs
    gaps_in_samples = np.round(inter_pulse_ms / sample_ms)
    bin_edges = (np.arange(gaps_in_samples.min(), gaps_in_samples.max() + 2) - 0.5) * sample_ms
    first_5s = int(5 * fs)
    fig, ax = plt.subplots(1, 2, figsize=(13, 3))
    ax[0].step(side_data["time"][:first_5s] / 1000, side_data["digital_1"][:first_5s], where="post")
    ax[0].set_title(f"digital_1 rsync TTL (first 5 s) — {side_data['Full_side_name']}")
    ax[0].set_xlabel("time (s)"); ax[0].set_ylabel("TTL")
    ax[1].hist(inter_pulse_ms, bins=bin_edges)
    ax[1].set_title(f"rsync inter-pulse interval (ms), n={len(inter_pulse_ms)}")
    ax[1].set_xlabel("ms (86 Hz sample grid)")
    fig.tight_layout()

## Lick detection

`LickBinary_2.3` is the per-sample binary lick at the photometry clock. `RawLickData` is the full per-video-frame acquisition table the licks were derived from.

In [ ]:
for side_data in data:
    lick_binary = side_data["LickBinary_2.3"]
    raw_lick_df = side_data["RawLickData"]
    print(f"{side_data['Full_side_name']}: LickBinary unique {np.unique(lick_binary[~np.isnan(lick_binary)])} | "
          f"n licks(1s) {int(np.nansum(lick_binary))} | n NaN {int(np.isnan(lick_binary).sum())} | "
          f"RawLickData {raw_lick_df.shape}")

# Structural preview (columns are the same for both sides)
print("\nRawLickData columns:", list(data[0]["RawLickData"].columns))
data[0]["RawLickData"].head()

In [ ]:
# RawLickData column meanings:
#   AbsTime / Abs_time2 / True_Absolute_Time : wall-clock timestamps (raw, +offset, rsync-corrected)
#   RelTime : ms since acquisition start    ms_since_midnight / True_Time_ms : clock-based time
#   LixPos  : spout/lixit position           AnalogLick : raw lick-sensor voltage
#   VDiff   : voltage difference (lick-detection feature)   BinLick : detected lick (0/1)
#   rSync   : rsync TTL on the lick stream
for side_data in data:
    raw_lick_df = side_data["RawLickData"]
    print(side_data["Full_side_name"], "numeric summary:")
    print(raw_lick_df.select_dtypes("number").describe().T[["min", "max", "mean"]], "\n")

    middle = len(raw_lick_df) // 2                  # a window in the middle of the table
    window_df = raw_lick_df.iloc[middle:middle + 2000]
    fig, ax = plt.subplots(4, 1, figsize=(13, 8), sharex=True)
    ax[0].plot(window_df["RelTime"] / 1000, window_df["AnalogLick"], lw=0.7); ax[0].set_ylabel("AnalogLick (V)")
    ax[1].plot(window_df["RelTime"] / 1000, window_df["VDiff"], lw=0.7, color="tab:purple"); ax[1].set_ylabel("VDiff")
    ax[2].step(window_df["RelTime"] / 1000, window_df["BinLick"], where="mid", color="k"); ax[2].set_ylabel("BinLick")
    ax[3].plot(window_df["RelTime"] / 1000, window_df["LixPos"], lw=0.7, color="tab:blue"); ax[3].set_ylabel("LixPos")
    ax[3].set_xlabel("RelTime (s)")
    ax[0].set_title(f"RawLickData — sensor, vdiff, detected licks, spout position — {side_data['Full_side_name']}")
    fig.tight_layout()

## Lick bursts

`LickBurst_Vars_BurstDefinitionILI_basedThresh2000` — bursts defined by inter-lick intervals with a 2000 ms gap threshold. Contains per-lick, per-burst and binned-rate arrays, plus the per-sample burst labeling.

In [ ]:
# A no-lick session only carries NumLicks / CumLicks / Labeled_BurstLick in burst_vars;
# the per-burst stat keys are absent, so default them.
for side_data in data:
    burst_vars = side_data[BURST_KEY]
    num_bursts = burst_vars.get("NumBursts", 0)
    threshold = burst_vars.get("BurstThreshold_ms", "n/a")
    avg_licks = np.mean(burst_vars["Avg_LicksPerBurst"]) if num_bursts else 0
    print(f"{side_data['Full_side_name']}: NumLicks={burst_vars['NumLicks']}  NumBursts={num_bursts}  "
          f"BurstThreshold={threshold} ms  mean licks/burst={avg_licks:.0f}")

# field listing (a no-lick side lists only the few keys it has)
print("\nfields:")
for key, value in data[0][BURST_KEY].items():
    print(f"  {key:24s} {describe(value)}")

In [ ]:
for side_data in data:
    burst_vars = side_data[BURST_KEY]
    if burst_vars["NumLicks"] == 0:
        print(f"{side_data['Full_side_name']}: no licks recorded, skipping burst-structure plots")
        continue
    fig, ax = plt.subplots(2, 2, figsize=(13, 7))
    # Lick durations are quantized to the 86 Hz sample (~11.6 ms), so scale x to the data.
    lick_durations = np.asarray(burst_vars["LickDurations_ms"], dtype=float)
    duration_hi = float(np.nanpercentile(lick_durations, 99)) * 1.5
    ax[0, 0].hist(lick_durations, bins=60, range=(0, duration_hi)); ax[0, 0].set_title("Lick durations (ms)"); ax[0, 0].set_xlim(0, duration_hi)
    ax[0, 1].hist(np.clip(burst_vars["InterlickInterval_ms"], 0, 500), bins=80); ax[0, 1].set_title("Inter-lick intervals (ms, clipped 500)")
    ax[1, 0].hist(burst_vars["Avg_LicksPerBurst"], bins=30); ax[1, 0].set_title("Licks per burst")
    # Full_BurstDur is Lick_BurstDur + one sample, so they overlay; show just the lick-burst duration.
    ax[1, 1].hist(np.array(burst_vars["Lick_BurstDur"]) / 1000, bins=30)
    ax[1, 1].set_title("Lick burst durations (s)")
    fig.suptitle(side_data["Full_side_name"]); fig.tight_layout()

In [ ]:
# Per-sample burst labeling (Labeled_BurstLick) and within-burst ILIs.
# These session-cropped arrays begin at SessionStart_frameNum on the photometry clock.
# A no-lick session has all-zero CumLicks / burst ids and no ILI_withinBursts.
for side_data in data:
    fs = side_data["sampling_rate"]
    burst_vars = side_data[BURST_KEY]
    labeled_burst_lick = np.asarray(burst_vars["Labeled_BurstLick"])  # cols: (in-burst flag, burst id)
    burst_time_min = (side_data["SessionStart_frameNum"] + np.arange(len(labeled_burst_lick))) / fs / 60
    within_bursts = [np.asarray(x) for x in burst_vars.get("ILI_withinBursts", []) if len(np.asarray(x)) > 0]
    within_burst_ilis = np.concatenate(within_bursts) if within_bursts else np.array([])

    fig, ax = plt.subplots(3, 1, figsize=(13, 7))
    ax[0].plot(burst_time_min, np.asarray(burst_vars["CumLicks"])); ax[0].set_ylabel("cum licks")
    ax[0].set_title(f"Cumulative licks — {side_data['Full_side_name']}")
    ax[0].set_xlabel("time (min)")
    ax[1].plot(burst_time_min, labeled_burst_lick[:, 1], lw=0.6); ax[1].set_ylabel("burst id")
    ax[1].set_xlabel("time (min)"); ax[1].set_title("Labeled_BurstLick (burst id per sample)")
    ax[2].hist(within_burst_ilis, bins=50, color="steelblue")
    ax[2].set_xlabel("Within-burst ILI (ms)")
    ax[2].set_ylabel("Count")
    ax[2].set_title("Within-burst inter-lick intervals")
    fig.tight_layout()

## 7. Lick rate at 1 s / 1 min / 5 min bins

In [ ]:
for side_data in data:
    burst_vars = side_data[BURST_KEY]
    rate_specs = [("Lickrate_1s", "1 s", 1 / 60), ("Lickrate_1m", "1 min", 1), ("Lickrate_5m", "5 min", 5)]
    # A no-lick session omits the lick-rate arrays.
    if not all(rate_key in burst_vars for rate_key, _, _ in rate_specs):
        print(f"{side_data['Full_side_name']}: no lick-rate series (no licks recorded), skipping")
        continue
    fig, ax = plt.subplots(3, 1, figsize=(13, 6), sharex=True)
    for axis, (rate_key, label, bin_min) in zip(ax, rate_specs):
        rate = np.asarray(burst_vars[rate_key])
        axis.plot(np.arange(len(rate)) * bin_min, rate, lw=0.8)
        axis.set_ylabel("licks/min"); axis.set_title(f"Lick rate ({label} bins), len={len(rate)}")
    ax[-1].set_xlabel("time (min)"); fig.suptitle(side_data["Full_side_name"]); fig.tight_layout()

## 8. Bottle position & trial structure

`BottlePos` marks when the sucrose spout is available. Each access period is a 'trial' — the task is a baseline period followed by repeated access / no-access periods.

In [ ]:
for side_data in data:
    fs = side_data["sampling_rate"]
    time_min = side_data["time"] / 1000 / 60
    bottle_in = np.nan_to_num(side_data["BottlePos"]) > 0.5
    access_starts = np.where(bottle_in[1:] & ~bottle_in[:-1])[0] + 1   # rising edges = spout in
    access_ends = np.where(~bottle_in[1:] & bottle_in[:-1])[0] + 1     # falling edges = spout out
    n_access = min(len(access_starts), len(access_ends))
    access_durations_s = (access_ends[:n_access] - access_starts[:n_access]) / fs
    print(f"{side_data['Full_side_name']}: {len(access_starts)} access periods | "
          f"mean duration {np.mean(access_durations_s):.1f} s | "
          f"BottleIn_frameNum {side_data['BottleIn_frameNum']} ({side_data['BottleIn_frameNum']/fs:.1f} s)")

    fig, ax = plt.subplots(figsize=(13, 2.4))
    for start_idx, end_idx in zip(access_starts[:n_access], access_ends[:n_access]):
        ax.axvspan(start_idx / fs / 60, end_idx / fs / 60, color="tab:blue", alpha=0.25, lw=0)
    ax.set_xlim(time_min[0], time_min[-1]); ax.set_ylim(0, 1); ax.set_yticks([])
    ax.set_title(f"BottlePos access periods ({len(access_starts)} trials) — {side_data['Full_side_name']}")
    ax.set_xlabel("time (min)")
    fig.tight_layout()

## 9. DLC tracking — head/nose-to-spout distances

DeepLabCut-derived **distances** (not raw x/y) plus likelihoods, interpolated to the photometry clock. `Cleaned_Head_Distance` is the jump-corrected, session-cropped version.

In [ ]:
for side_data in data:
    time_min = side_data["time"] / 1000 / 60
    dlc_keys = [key for key in side_data if key.startswith("DLC_")]
    print(side_data["Full_side_name"], "DLC keys:", dlc_keys)

    fig, ax = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
    ax[0].plot(time_min, side_data["DLC_Distance_head_middle-spout"], lw=0.6, label="head_middle-spout")
    ax[0].plot(time_min, side_data["DLC_Distance_nose-spout"], lw=0.6, label="nose-spout")
    ax[0].plot(time_min, side_data["DLC_Distance_head_middle-nose"], lw=0.6, label="head-nose", alpha=0.6)
    ax[0].set_ylabel("distance (px)"); ax[0].legend(fontsize=8)
    ax[0].set_title(f"DLC distances — {side_data['Full_side_name']}")
    ax[1].plot(time_min, side_data["DLC_Likelihood_head_middle"], lw=0.6, label="head_middle")
    ax[1].plot(time_min, side_data["DLC_Likelihood_nose"], lw=0.6, label="nose")
    ax[1].set_ylabel("likelihood"); ax[1].set_xlabel("time (min)"); ax[1].legend(fontsize=8); ax[1].set_title("DLC likelihoods")
    fig.tight_layout()

## 10. Engagement state vectors

Binary 'engaged with spout' vectors — an automatic threshold and several manual ones, for both head- and nose-based distance. The number in each key name is the distance threshold used.

In [ ]:
for side_data in data:
    time_min = side_data["time"] / 1000 / 60
    engagement_keys = [key for key in side_data if key.startswith("Engagement")]
    fig, ax = plt.subplots(figsize=(13, 4))
    for row, engagement_key in enumerate(engagement_keys):
        engaged_fraction = float(np.mean(side_data[engagement_key]))
        print(f"  {side_data['Full_side_name']:22s} {engagement_key:42s} engaged fraction = {engaged_fraction:.3f}")
        ax.fill_between(time_min, row, row + np.asarray(side_data[engagement_key]) * 0.9, step="mid", lw=0)
    ax.set_yticks(np.arange(len(engagement_keys)) + 0.45); ax.set_yticklabels(engagement_keys, fontsize=7)
    ax.set_xlabel("time (min)"); ax.set_title(f"Engagement vectors (raster) — {side_data['Full_side_name']}")
    fig.tight_layout()

## 11. Distance states & approach/leave events

`Distance_States_Events` discretizes the head-to-spout distance into states (near/transition/far) and marks approach and leave transition events. The detection parameters live in `QC['distance_states_transition_events']`.

In [ ]:
for side_data in data:
    fs = side_data["sampling_rate"]
    distance_states = side_data["Distance_States_Events"]
    print(f"{side_data['Full_side_name']}: state unique {np.unique(distance_states['state'])} | "
          f"#approach {int((distance_states['Approach_events'] > 0).sum())} | "
          f"#leave {int((distance_states['Leave_events'] > 0).sum())}")

    # session-cropped arrays begin at SessionStart_frameNum on the photometry clock
    state_time_min = (side_data["SessionStart_frameNum"] + np.arange(len(distance_states["state"]))) / fs / 60
    approach_times_min = state_time_min[distance_states["Approach_events"] > 0]
    leave_times_min = state_time_min[distance_states["Leave_events"] > 0]
    fig, ax = plt.subplots(figsize=(13, 2.6))
    ax.plot(state_time_min, distance_states["state"], lw=0.6)
    ax.plot(approach_times_min, np.full_like(approach_times_min, 2.1), "^", ms=4, color="g", label="approach")
    ax.plot(leave_times_min, np.full_like(leave_times_min, -0.1), "v", ms=4, color="r", label="leave")
    ax.set_xlabel("time (min)"); ax.set_ylabel("state"); ax.legend(fontsize=8)
    ax.set_title(f"Distance state + transitions — {side_data['Full_side_name']}")
    fig.tight_layout()

print("\nDetection settings (QC['distance_states_transition_events']):")
for key, value in data[0]["QC"]["distance_states_transition_events"].items():
    print(f"    {key:20s} {value}")

## 12. QC — Hampel outlier removal

Parameters and outcome of the Hampel filter applied to each analog channel.

In [ ]:
for side_data in data:
    hampel_qc = side_data["QC"]["hampel"]
    print(side_data["Full_side_name"])
    print("  params:       ", {k: v for k, v in hampel_qc.items() if not isinstance(v, dict)})
    print("  n_outliers:   ", hampel_qc["n_outliers"])
    print("  frac_outliers:", {k: round(v, 6) for k, v in hampel_qc["frac_outliers"].items()})

    # Find the channel + sample where the Hampel filter made the largest correction
    worst_channel, worst_idx, worst_correction = None, 0, -1.0
    for raw_key in ("analog_1", "analog_2", "analog_3"):
        hampel_key = f"analog{raw_key[-1]}_hampel"
        correction = np.abs(side_data[raw_key] - side_data[hampel_key])
        peak_idx = int(np.argmax(correction))
        if correction[peak_idx] > worst_correction:
            worst_channel, worst_idx, worst_correction = raw_key, peak_idx, correction[peak_idx]

    start_idx, end_idx = max(0, worst_idx - 200), worst_idx + 200
    hampel_key = f"analog{worst_channel[-1]}_hampel"
    fig, ax = plt.subplots(figsize=(13, 3))
    ax.plot(side_data["time"][start_idx:end_idx] / 1000, side_data[worst_channel][start_idx:end_idx], label=f"{worst_channel} raw", lw=0.8)
    ax.plot(side_data["time"][start_idx:end_idx] / 1000, side_data[hampel_key][start_idx:end_idx], label="hampel", lw=0.8)
    ax.set_title(f"Largest raw-vs-hampel correction ({worst_channel}) — {side_data['Full_side_name']}")
    ax.set_xlabel("time (s)"); ax.set_ylabel("V"); ax.legend(fontsize=8); fig.tight_layout()

## 13. Wall-clock timing & frame markers

How the session sits in real time, and the key frame indices on the photometry clock.

In [ ]:
for side_data in data:
    fs = side_data["sampling_rate"]
    print("=" * 70, "\n", side_data["Full_side_name"])
    print("  date_time (start):", side_data["date_time"], "| end_time:", side_data["end_time"])
    print("  Abs_Time:         ", side_data["Abs_Time"].iloc[0], "->", side_data["Abs_Time"].iloc[-1])
    print("  ms_since_midnight:", int(side_data["ms_since_midnight"].iloc[0]), "->", int(side_data["ms_since_midnight"].iloc[-1]))
    for frame_key in ["BottleIn_frameNum", "SessionStart_frameNum", "SessionEnd_frameNum"]:
        frame = side_data[frame_key]
        print(f"    {frame_key:22s} {frame:>8d}  ({frame/fs:7.1f} s, {frame/fs/60:5.2f} min)")
    n_samples = len(side_data["analog_1"])
    print(f"    {'total samples':22s} {n_samples:>8d}  ({n_samples/fs/60:5.2f} min)")

## 14. Combined session overview

Everything aligned on the session clock — a 5 min window mid-session showing how photometry, licking, bottle availability, engagement and distance state line up.

In [ ]:
for side_data in data:
    fs = side_data["sampling_rate"]
    time_min = side_data["time"] / 1000 / 60
    distance_states = side_data["Distance_States_Events"]
    state_time_min = (side_data["SessionStart_frameNum"] + np.arange(len(distance_states["state"]))) / fs / 60
    auto_engagement_key = [key for key in side_data
                           if key.startswith("Engagement") and "auto" in key and "head" in key][0]

    window_start_min = time_min[-1] / 2                       # 5 min window centered mid-session
    window_end_min = min(window_start_min + 5, time_min[-1])
    in_window = (time_min >= window_start_min) & (time_min <= window_end_min)
    state_in_window = (state_time_min >= window_start_min) & (state_time_min <= window_end_min)

    fig, ax = plt.subplots(6, 1, figsize=(13, 10), sharex=True)
    ax[0].plot(time_min[in_window], side_data["analog2_hampel"][in_window], color="#D62728", lw=0.7); ax[0].set_ylabel("rDA (V)")
    ax[1].plot(time_min[in_window], side_data["analog1_hampel"][in_window], color="#2CA02C", lw=0.7); ax[1].set_ylabel("gACh4h (V)")
    ax[2].fill_between(time_min[in_window], 0, np.nan_to_num(side_data["LickBinary_2.3"])[in_window], step="mid", color="k"); ax[2].set_ylabel("lick")
    ax[3].fill_between(time_min[in_window], 0, np.nan_to_num(side_data["BottlePos"])[in_window], step="mid", color="tab:blue"); ax[3].set_ylabel("bottle")
    ax[4].fill_between(time_min[in_window], 0, side_data[auto_engagement_key][in_window], step="mid", color="tab:orange"); ax[4].set_ylabel("engaged")
    ax[5].plot(state_time_min[state_in_window], distance_states["state"][state_in_window], lw=0.8, color="tab:purple"); ax[5].set_ylabel("state")
    ax[5].set_xlabel("time (min)")
    fig.suptitle(f"Session overview ({window_start_min:.0f}-{window_end_min:.0f} min) — {side_data['Full_side_name']}")
    fig.tight_layout()

## 15. Compare the two sides side-by-side

In [ ]:
summary_rows = []
for index, side_data in enumerate(data):
    burst_vars = side_data[BURST_KEY]
    summary_rows.append({
        "index": index,
        "side": side_data["Full_side_name"],
        "filename": side_data["filename"],
        "n_samples": len(side_data["analog_1"]),
        "grams_consumed": side_data["GramConsumed"],
        "num_licks": burst_vars["NumLicks"],
        "num_bursts": burst_vars.get("NumBursts", 0),
        "session_start_frame": side_data["SessionStart_frameNum"],
        "session_end_frame": side_data["SessionEnd_frameNum"],
    })
pd.DataFrame(summary_rows).set_index("index")